In [3]:
def get_dataset_weights(good_datasets, garbage_datasets, good_percentage):
    """
    Calculate sampling weights for mixed quality datasets, compensating for size-based sampling.
    The final sampling probability will be weight * dataset_size, so we adjust weights accordingly.
    Returns weights that sum to exactly 1.
    
    Args:
        good_datasets: Dict of good dataset paths and their sizes in bytes
        garbage_datasets: Dict of garbage dataset paths and their sizes in bytes
        good_percentage: Percentage of time to sample from good datasets (0-100)
        
    Returns:
        dict: Dataset paths and their corresponding weights that sum to 1
    """
    # Convert percentage to fraction
    good_fraction = good_percentage / 100.0
    garbage_fraction = 1.0 - good_fraction
    
    weights = {}
    
    # For good datasets, we want them sampled equally after size multiplication
    # So initial weight = target_prob / size
    for dataset, size in good_datasets.items():
        target_prob = good_fraction / len(good_datasets)
        weights[dataset] = target_prob / size
        
    # For garbage datasets, we want their final probabilities proportional to sizes
    total_garbage_size = sum(garbage_datasets.values())
    for dataset in garbage_datasets:
        weights[dataset] = garbage_fraction / total_garbage_size
    
    # Simple normalization to make weights sum to 1
    total_weight = sum(weights.values())
    for dataset in weights:
        weights[dataset] /= total_weight
        
    return weights

# Example usage with the provided datasets
good_datasets = {
    "sawyer__pn_p_counter_to_cab_aux": 496e6,
    "sawyer__pn_p_counter_to_sink_aux": 397e6,
    "sawyer__pn_p_sink_to_counter_aux": 552e6
}

garbage_datasets = {
    # "ghost__kinova3_mimicgen": 3.9e9,
    # "ghost__panda_mimicgen": 4.0e9,
    # "ghost__sawyer_mimicgen": 4.5e9,
    # "ghost_u_r5e_mimicgen": 3.2e9,
    # "solid__kinova3_mimicgen": 6.0e9,
    # "solid__panda_mimicgen": 4.1e9,
    "solid__sawyer_mimicgen": 7.7e9,
    # "solid_u_r5e_mimicgen": 4.0e9
}

# Get weights with 30% sampling from good datasets
weights = get_dataset_weights(good_datasets, garbage_datasets, 50)

# Verify weights sum to 1
print(f"Sum of weights: {sum(weights.values()):.10f}")

# Print the weights and expected sampling probabilities
print("\nWeights and probabilities after size multiplication:")
for dataset, weight in weights.items():
    size = good_datasets.get(dataset) or garbage_datasets.get(dataset)
    true_prob = weight * size
    print(f"{dataset}:")
    print(f"  Weight: {weight:.2e}")
    print(f"  True sampling probability: {true_prob:.1%}")

Sum of weights: 1.0000000000

Weights and probabilities after size multiplication:
sawyer__pn_p_counter_to_cab_aux:
  Weight: 2.99e-01
  True sampling probability: 14845109146.7%
sawyer__pn_p_counter_to_sink_aux:
  Weight: 3.74e-01
  True sampling probability: 14845109146.7%
sawyer__pn_p_sink_to_counter_aux:
  Weight: 2.69e-01
  True sampling probability: 14845109146.7%
solid__sawyer_mimicgen:
  Weight: 5.78e-02
  True sampling probability: 44535327440.2%


In [4]:
weights

{'sawyer__pn_p_counter_to_cab_aux': 0.2992965553779019,
 'sawyer__pn_p_counter_to_sink_aux': 0.37393222032100587,
 'sawyer__pn_p_sink_to_counter_aux': 0.26893313671637564,
 'solid__sawyer_mimicgen': 0.05783808758471663}